# Step 06 — Build the validation set

**Input**

- `data/validation/sample_version_1.xlsx` — the hand-annotated workbook. The one

  irreplaceable file in this branch; everything below is derived from it.

- `config.VALIDATION_BUILDINGS_FILE` — the *frozen* condensed buildings file from

  the ORIGINAL pipeline run, read-only, outside this repo.

- `data/output/05_condensed_buildings_with_pois.gpkg` — the CURRENT run, used only

  to recover a run-stable id.

**Output** — `data/validation/06_validation_set.csv`, the single input to

notebooks 07–10.

---

## What the workbook is

An earlier LLM run's predictions, printed to a spreadsheet, which a human then

went through cell by cell. The verdict is encoded as a **cell fill colour**:

| Colour | Meaning |

|---|---|

| green | the prediction was accepted as correct |

| red | the prediction was wrong — the corrected value is typed in the neighbouring `_new` / `.1` column |

| yellow / uncoloured | the validator could not decide — **not scoreable**, dropped |

Keeping only rows that are red-or-green on *both* dimensions leaves **889** of

1,391 buildings.

## Why this notebook also repairs the id

`gml_id` in the workbook is a **positional row index** minted by notebook 05 of the

original run. It is not reproducible: re-running 05 renumbers everything

(578,080 buildings then, 574,435 now). Joining the workbook to a fresh run on

`gml_id` succeeds for ~95% of rows and every one of them is a *different*

building — the join looks healthy and silently compares unrelated things.

The fix is to go through **geometry**, which never moved:

```

workbook gml_id → frozen-run footprint → point-in-polygon → register id

```

The register id (`source_gml_id`) is the ALKIS cadastral id, or `osm_<id>` for a

building that exists only in OpenStreetMap. It comes from a source register, not

from our row ordering, so it is stable across runs and is the key notebooks 07-10

join on.

The validation set therefore carries exactly **two** ids and no more:

| column | what it is | use it for |

|---|---|---|

| `gml_id` | the workbook's row id | identifying a row **inside** this set |

| `source_gml_id` | register id of the current-run building | **joining** to pipeline output |

Both are needed. `source_gml_id` is the only safe join key, but it is not unique

across these rows — notebook 05 merges buildings that were annotated separately —

so it cannot also be the row identity.

In [ ]:
import sys

sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

import numpy as np

import pandas as pd

import geopandas as gpd

from openpyxl import load_workbook

from config import (VALIDATION_SOURCE_FILE, VALIDATION_BUILDINGS_FILE, VALIDATION_SET_FILE,

                    CONDENSED_BUILDINGS_FILE, TARGET_CRS, VALIDATION_COLOURS)

pd.set_option('display.width', 200)

pd.set_option('display.max_colwidth', 60)

print('Config loaded')

## 1. Decode the colour-coded workbook

`openpyxl` reports fill colours as ARGB hex; `config.VALIDATION_COLOURS` maps the

three Excel standard fills to their meaning. A cell with no fill returns `None`.

In [ ]:
raw = pd.read_excel(VALIDATION_SOURCE_FILE, sheet_name=0)

ws  = load_workbook(VALIDATION_SOURCE_FILE).worksheets[0]

def fill_of(cell):

    f = cell.fill

    if f is None or f.patternType is None:

        return None

    s = f.start_color

    if s.type == 'rgb' and s.rgb and s.rgb != '00000000':

        return s.rgb[-6:].upper()

    return None

# VALIDATION_COLOURS is keyed on the 8-digit ARGB openpyxl usually reports; match

# on the last 6 (RGB) so a fill written as either form resolves the same way.

rgb_to_meaning = {k[-6:].upper(): v for k, v in VALIDATION_COLOURS.items()}

fills = pd.DataFrame([[fill_of(c) for c in row]

                      for row in ws.iter_rows(min_row=2, max_col=len(raw.columns))])

fills = fills.reindex(index=range(len(raw)), columns=range(len(raw.columns)))

fills.columns = raw.columns

for col in ['mid_label_new', 'bosserhof_class_clean.1']:

    raw[f'{col}_color'] = fills[col].map(rgb_to_meaning)

print(f'{len(raw):,} rows in the workbook')

print(raw['mid_label_new_color'].value_counts(dropna=False).to_string())

print(raw['bosserhof_class_clean.1_color'].value_counts(dropna=False).to_string())

## 2. Rename to speaking columns and keep only decided rows

A row is scoreable only where the validator actually rendered a verdict. Yellow

means "could not decide" and an uncoloured cell means "never looked at" — neither

is a ground truth, and including them would score the classifier against a blank.

In [ ]:
df = raw.rename(columns={

    'mid_label':                     'Predicted_activities',

    'mid_label_new':                 'Predicted_activities_mistakes',

    'mid_label_new_color':           'Predicted_activities_mistakes_color',

    'bosserhof_class_clean':         'Bosserhof_class_predicted',

    'bosserhof_class_clean.1':       'Bosserhof_class_mistakes',

    'bosserhof_class_clean.1_color': 'Bosserhof_class_mistakes_color',

})[[

    'gml_id', 'osm_names', 'volume_m3',

    'Predicted_activities', 'Predicted_activities_mistakes', 'Predicted_activities_mistakes_color',

    'Bosserhof_class_predicted', 'Bosserhof_class_mistakes', 'Bosserhof_class_mistakes_color',

]]

verdict_cols = ['Predicted_activities_mistakes_color', 'Bosserhof_class_mistakes_color']

df = df[df[verdict_cols].isin(['red', 'green']).all(axis=1)].reset_index(drop=True)

df['gml_id'] = pd.to_numeric(df['gml_id'], errors='coerce').astype('Int64')

assert df['gml_id'].is_unique, 'the workbook has duplicate gml_id values'

# Guard the count, not just the shape. Every assertion further down is over rows,

# so on an empty frame they all pass vacuously and this notebook writes a valid,

# empty CSV without raising. That is exactly what happened when the colour map

# resolved to 'correct'/'error' instead of 'green'/'red': 0 rows, 0 errors, and a

# silently useless validation set. A floor makes the failure loud.

assert len(df) >= 800, (

    f'only {len(df)} rows survived the verdict filter — expected ~889. The colour '

    f'decode probably failed; observed values: '

    f'{sorted(set(raw["mid_label_new_color"].dropna()))}')

print(f'{len(df):,} rows decided on BOTH dimensions')

for c in verdict_cols:

    print(f'  {c:38s} {df[c].value_counts().to_dict()}')

## 3. Attach the evidence text from the frozen run

`sentence` is the rendered prompt — the same field set, in the same order, that

the earlier LLM run saw. Notebook 08 sends it to the model verbatim, so it must be

built from the file the workbook was sampled from, not from a fresh run.

Two guards, because a bad join here does not raise, it returns *wrong rows*:

`validate='one_to_one'` catches a duplicated key, and the volume assertion catches

a key that is unique but points at a different building.

In [ ]:
frozen = gpd.read_file(VALIDATION_BUILDINGS_FILE)

frozen['gml_id'] = pd.to_numeric(frozen['gml_id'], errors='coerce').astype('Int64')

print(f'{len(frozen):,} buildings in the frozen run  (crs={frozen.crs})')

PRECISE = [('osm_names', 'name'), ('amenity', 'amenity'), ('building', 'building'),

           ('shop', 'shop'), ('tourism', 'tourism'), ('information', 'information'),

           ('website', 'website'), ('email', 'email')]

GENERAL = [('label_en', 'building_label'), ('osm_building_type', 'osm_building_type'),

           ('osm_landuse_class', 'osm_landuse_class'), ('osm_landuse_name', 'osm_landuse_name'),

           ('gfk_class', 'gfk_class'), ('ALKIS_Landuse_info', 'alkis_landuse'),

           ('tags_search', 'tags'), ('additional_information', 'additional_info')]

BAD = {'', 'nan', 'none', 'null', 'na', 'n/a', '<na>', '-', '--'}

def clean(v):

    if v is None or v is pd.NA:

        return None

    if isinstance(v, float) and np.isnan(v):

        return None

    s = str(v).strip()

    return None if s.lower() in BAD else s

def render_sentence(row):

    out = []

    for fields, header in ((PRECISE, 'precise_known_info'), (GENERAL, 'general_building_context')):

        bits = [f'{label}: {clean(row.get(col))}' for col, label in fields

                if clean(row.get(col)) is not None]

        if bits:

            out.append(f'{header}: ' + ' | '.join(bits))

    return '\n'.join(out)

frozen['sentence'] = frozen.apply(render_sentence, axis=1)

merged = df.merge(frozen[['gml_id', 'volume_m3', 'sentence']], on='gml_id',

                  how='left', suffixes=('', '_frozen'), validate='one_to_one')

assert merged['sentence'].notna().all(), 'a workbook row has no match in the frozen file'

agree = np.isclose(merged['volume_m3'], merged['volume_m3_frozen'], rtol=1e-9, atol=1e-6)

assert agree.all(), (f'{(~agree).sum()} rows disagree on volume — VALIDATION_BUILDINGS_FILE '

                     'is not the run this workbook was annotated against')

merged = merged.drop(columns='volume_m3_frozen')

print(f'joined {len(merged):,} rows, volume identical on all of them')

## 4. Attach a run-stable `source_gml_id`

One spatial join: drop a point inside each annotated footprint and ask which

building in the **current** run contains it, then take that building's

`source_gml_id`. A *representative* point rather than a centroid — a centroid can

fall outside its own polygon for an L-shaped or courtyard building.

That single lookup answers both questions at once: which building in today's

pipeline this annotation refers to, and what its stable register id is. An earlier

version resolved the id against notebook 01's pre-merge ALKIS polygons and then

carried a second positional column to find the building again; going straight to

05 gives a byte-identical answer for all 889 rows with one join and one id.

The resulting id is an ALKIS cadastral id (`DENILD01000002A1`) or `osm_<id>` for a

building that exists only in OpenStreetMap. Either way it comes from a source

register rather than from our row ordering, so it survives a re-run — which

`gml_id` does not.

In [ ]:
foot = frozen[frozen['gml_id'].isin(merged['gml_id'])].copy()

if foot.crs is not None and foot.crs.to_string() != TARGET_CRS:

    foot = foot.to_crs(TARGET_CRS)

pts = foot[['gml_id', 'geometry']].copy()

pts['geometry'] = pts.geometry.representative_point()

cur = gpd.read_file(CONDENSED_BUILDINGS_FILE, columns=['gml_id', 'source_gml_id'])

if cur.crs is not None and cur.crs.to_string() != TARGET_CRS:

    cur = cur.to_crs(TARGET_CRS)

print(f'{len(cur):,} buildings in the current run')

hit = gpd.sjoin(pts, cur[['source_gml_id', 'geometry']], how='left',

                predicate='within').drop(columns='index_right')

n_multi = int((hit.groupby('gml_id').size() > 1).sum())

assert n_multi == 0, (f'{n_multi} footprints fall inside more than one current building — '

                      'the current run has overlapping polygons and the id would be arbitrary')

id_map = hit[['gml_id', 'source_gml_id']].drop_duplicates('gml_id')

n_lost = int(id_map['source_gml_id'].isna().sum())

assert n_lost == 0, (f'{n_lost} annotated footprints are inside no building in the current '

                     'run — they were filtered out by a threshold change in 01-05')

print(f'resolved {len(id_map):,}/{len(pts):,} to a register id')

### Register ids shared by more than one annotated row

Not an error, and deliberately not deduplicated. Notebook 05 merges touching

polygons, and the current run merges harder than the original did, so a few

footprints that were annotated separately are one building today: "Pension Haus

Trautheim" was three LOD2 fragments of 19/11/114 m² and is one 150 m² building

here.

This is why `gml_id` is still carried. `source_gml_id` is the right key to *join*

on but it is not unique across these 889 rows, so it cannot also serve as the row

identity — keying on it would silently collapse 889 annotations into 882.

Collapsing them is not safe anyway: most groups agree internally, but two do not

(*public facilities* vs *normal office*, *restaurants gastronomy* vs *retail small

scale*), so merging would mean arbitrarily discarding one human verdict.

In [ ]:
dup = id_map['source_gml_id'].duplicated(keep=False)

if dup.any():

    print(f"{int(dup.sum())} rows share a register id, in "

          f"{id_map.loc[dup, 'source_gml_id'].nunique()} groups "

          f'(kept as separate rows — see above)')

else:

    print('every register id is unique to one annotated row')

## 5. Write the validation set

Two id columns, and only two:

| column | what it is | use it for |

|---|---|---|

| `gml_id` | the workbook's row id | identifying a row **inside** this validation set |

| `source_gml_id` | the register id of the current-run building | **joining** to any pipeline output |

Never join `gml_id` against a pipeline file — that is the whole failure this

notebook exists to undo.

In [ ]:
out = merged.merge(id_map, on='gml_id', how='left', validate='one_to_one')

assert out['source_gml_id'].notna().all()

assert out['gml_id'].is_unique, 'gml_id must stay unique — it is the row identity'

assert len(out) == len(merged)

VALIDATION_SET_FILE.parent.mkdir(parents=True, exist_ok=True)

out.to_csv(VALIDATION_SET_FILE, index=False, encoding='utf-8')

print(f'wrote {VALIDATION_SET_FILE.name}  ({len(out):,} rows, {len(out.columns)} cols)')

print(f'columns: {list(out.columns)}')

out.head(3)